# 1. Własny Model

In [3]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD

# Needs to be fixed

# 1. Definicja struktury
# Wierzchołki: ['asia', 'tub', 'smoke', 'lung', 'bronc', 'either', 'xray', 'dysp']
# Krawędzie:   [('asia', 'tub'), ('tub', 'either'), ('smoke', 'lung'), ('smoke', 'bronc'), ('lung', 'either'), ('bronc', 'dysp'), ('either', 'xray'), ('either', 'dysp')]
asia_model = DiscreteBayesianNetwork(
    [
        ('asia', 'tub'),
        ('tub', 'either'),
        ('smoke', 'lung'),
        ('smoke', 'bronc'),
        ('lung', 'either'),
        ('bronc', 'dysp'),
        ('either', 'xray'),
        ('either', 'dysp')]
)

# 2. Definicja Prawdopodobieństwa Warunkowych (CDPs)
# variable_card -> ile możliwych stanów może przyjąć zmienna
from pgmpy.factors.discrete import TabularCPD

cdp_asia = TabularCPD(
    variable="asia",
    variable_card=2,
    values=[
        [0.01],  # yes
        [0.99]   # no
    ],
    state_names={"asia": ["yes", "no"]}
)

cdp_tub = TabularCPD(
    variable="tub",
    variable_card=2,
    # Columns: asia=[yes, no] | Rows: tub=[yes, no]
    values=[
        [0.01, 0.05],  # yes
        [0.99, 0.95]   # no
    ],
    evidence=["asia"],
    evidence_card=[2],
    state_names={
        "tub": ["yes", "no"],
        "asia": ["yes", "no"]
    }
)

cdp_smoke = TabularCPD(
    variable="smoke",
    variable_card=2,
    values=[
        [0.5],  # yes
        [0.5]   # no
    ],
    state_names={"smoke": ["yes", "no"]}
)

cdp_lung = TabularCPD(
    variable="lung",
    variable_card=2,
    # Columns: smoke=[yes, no] | Rows: lung=[yes, no]
    values=[
        [0.99, 0.9],  # yes
        [0.01, 0.1]   # no
    ],
    evidence=["smoke"],
    evidence_card=[2],
    state_names={
        "lung": ["yes", "no"],
        "smoke": ["yes", "no"]
    }
)

cdp_bronc = TabularCPD(
    variable="bronc",
    variable_card=2,
    # Columns: smoke=[yes, no] | Rows: bronc=[yes, no]
    values=[
        [0.7, 0.4],  # yes
        [0.3, 0.6]   # no
    ],
    evidence=["smoke"],
    evidence_card=[2],
    state_names={
        "bronc": ["yes", "no"],
        "smoke": ["yes", "no"]
    }
)

cdp_either = TabularCPD(
    variable="either",
    variable_card=2,
    # Columns order for parents (lung, tub):
    # [ (yes,yes), (yes,no), (no,yes), (no,no) ]
    values=[
        [1.0, 1.0, 1.0, 0.0],  # yes
        [0.0, 0.0, 0.0, 1.0]   # no
    ],
    evidence=["lung", "tub"],
    evidence_card=[2, 2],
    state_names={
        "either": ["yes", "no"],
        "lung": ["yes", "no"],
        "tub": ["yes", "no"]
    }
)

cdp_xray = TabularCPD(
    variable="xray",
    variable_card=2,
    # Columns: either=[yes, no] | Rows: xray=[yes, no]
    values=[
        [0.95, 0.02],  # yes
        [0.05, 0.98]   # no
    ],
    evidence=["either"],
    evidence_card=[2],
    state_names={
        "xray": ["yes", "no"],
        "either": ["yes", "no"]
    }
)

cdp_dysp = TabularCPD(
    variable="dysp",
    variable_card=2,
    # Columns order for parents (bronc, either):
    # [ (yes,yes), (yes,no), (no,yes), (no,no) ]
    values=[
        [0.9, 0.3, 0.2, 0.1],  # yes
        [0.1, 0.7, 0.8, 0.9]   # no
    ],
    evidence=["bronc", "either"],
    evidence_card=[2, 2],
    state_names={
        "dysp": ["yes", "no"],
        "bronc": ["yes", "no"],
        "either": ["yes", "no"]
    }
)

# 3. Dodanie prawdopodobieństw do modelu
asia_model.add_cpds(cdp_asia, cdp_tub, cdp_smoke, cdp_lung, cdp_bronc, cdp_either, cdp_xray, cdp_dysp)

# 4. sprawdzenie definicji modelu
asia_model.check_model()

True

## 1.1 Ocena Struktury

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx


pos = {
    "asia": (0, 3), "smoke": (2.5, 3),
    "tub": (0, 2), "lung": (1.7, 2), "bronc": (3.3, 2),
    "either": (0.8, 1),
    "xray": (0, 0), "dysp": (2.2, 0),
}

plt.figure(figsize=(6, 4.5))
nx.draw_networkx(asia_model, pos=pos, node_size=2200, node_color="#cfe2f3",
                 edgecolors="k", arrowsize=20, font_size=10)
plt.axis("off")
plt.title("Sieć bayesowska Asia")
plt.show()


## 1.2 Weryfikacja CDP

In [ ]:
for cpd in asia_model.get_cpds():
    print(cpd, end="\n\n")


## 1.3 Przykładowe Zapytania

In [ ]:
from pgmpy.example_models import load_model
from pgmpy.inference import VariableElimination

loaded_asia_model = load_model("bnlearn/asia")
loaded_asia_infer = VariableElimination(loaded_asia_model)

asia_infer = VariableElimination(asia_model)

queries = [
{"variables": ["bronc"], "evidence": {"smoke":"no"}},
{"variables": ["bronc", "asia"], "evidence": {"smoke":"yes"}},
{"variables": ["bronc", "asia"], "evidence": {"smoke":"no"}},
]

for idx, q in enumerate(queries, start=1):
    print(f"# {idx} query")
    print("Loaded Model")
    print(loaded_asia_infer.query(variables=q["variables"], evidence=q["evidence"]))
    print("My Model")
    print(asia_infer.query(variables=q["variables"], evidence=q["evidence"]))
    print()
    print()